In [10]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd
import numpy as np

In [8]:
dust_points_vars = pd.read_csv("DATA/processed/3_dust_points_vars_2026-04-21.csv")
non_dust_points_vars = pd.read_csv("DATA/processed/4_non_dust_points_vars_2026-04-21.csv")


In [7]:
def plot_bar_soil_texture(dust_df, non_dust_df):
    """
    Create a side-by-side bar chart comparing:
    - frequency of soil textures at dust points
    - frequency of soil textures in the full soil raster
    """

    texture_dict = {
        1: "Sand",
        2: "Loamy Sand",
        3: "Sandy Loam",
        4: "Silt Loam",
        5: "Silt",
        6: "Loam",
        7: "Sandy Clay Loam",
        8: "Silty Clay Loam",
        9: "Clay Loam", 
        10: "Sandy Clay",
        11: "Silty Clay",
        12: "Clay", 
        13: "Organic Matter",
        14: "Water", 
        15: "Bedrock",
        16: "Other",
    }

    #--- Calculate bins
    dust_counts = {k: np.sum(dust_df['texture'] == k) for k in texture_dict.keys()}
    dust_total = sum(dust_counts.values())
    dust_fraction = {k: v / dust_total for k, v in dust_counts.items()}

    non_dust_counts = {k: np.sum(non_dust_df['texture'] == k) for k in texture_dict.keys()}
    non_dust_total = sum(non_dust_counts.values())
    non_dust_fraction = {k: v / non_dust_total for k, v in non_dust_counts.items()}

    texture_colors = [
        "#EE6352",  # Sand
        "#e6d591",  # Loamy Sand
        "#d9c070",  # Sandy Loam
        "#c0b080",  # Silt Loam
        "#b0a070",  # Silt
        "#a67c52",  # Loam
        "#16DB93",  # Sandy Clay Loam
        "#9c6644",  # Silty Clay Loam
        "#805533",  # Clay Loam
        "#8c3f2f",  # Sandy Clay
        "#048BA8",  # Silty Clay
        "#4f1f18",  # Clay
        "#1a1a1a",  # Organic Matter
        "#3399ff",  # Water
        "#808080",  # Bedrock
        "#ffffff",  # Other
    ]

    # Prepare for plotting
    categories = list(texture_dict.keys())
    labels = [texture_dict[k] for k in categories]
    x = np.arange(len(categories))
    width = 0.4

    fig, ax = plt.subplots(figsize=(16, 8))

    # Plot bars
    for i, k in enumerate(categories):
        color = texture_colors[i]
        ax.bar(x[i] - width / 2, dust_fraction[k], width, color=color, edgecolor="black", label="Dust points" if i == 0 else "")
        ax.bar(x[i] + width / 2, non_dust_fraction[k], width, color=color, alpha=0.5, label="Full domain" if i == 0 else "")

    # Labels and ticks
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Fraction of observations", fontsize=18)
    ax.set_xlabel("Soil Texture", fontsize=18)
    ax.set_title(f"Soil Texture Frequency in American Southwest: Dust Points vs Full Domain \n (Wind Speeds >= 10 m/s)", fontsize=24)

    # Legend
    legend_elements = [
        Patch(facecolor="gray", edgecolor="black", label="Dust points"),
        Patch(facecolor="gray", edgecolor="black", alpha=0.5, label="Full domain")
    ]
    ax.legend(handles=legend_elements, title="Dataset")

    plt.tight_layout()
    plt.show()

    return

In [11]:
plot_bar_soil_texture(dust_points_vars, non_dust_points_vars)

KeyError: 'texture'

In [ ]:
def add_medians_to_plot(ax_bar, medians):

    ax_bar.axvline(
        medians[0],
        color="tab:orange",
        linestyle="--",
        linewidth=2,
        zorder=0
    )
    ax_bar.text(x=medians[0], 
                y=0.93,
                s=f'{medians[0]:.2f}', 
                color="tab:orange",
                alpha=0.8, 
                fontsize=10,
                fontweight='bold',
                rotation=90,
                verticalalignment='center',
                horizontalalignment='right',
                transform=ax_bar.get_xaxis_transform())

    ax_bar.axvline(
        medians[1],
        color="tab:blue",
        linestyle="--",
        linewidth=2,
        zorder=0
    )
    
    ax_bar.text(x=medians[1], 
                y=0.93,
                s=f'{medians[1]:.2f}', 
                color="tab:blue",
                alpha=0.8, 
                fontsize=10,
                fontweight='bold',
                rotation=90,
                verticalalignment='center',
                horizontalalignment='right',
                transform=ax_bar.get_xaxis_transform())
    
    return

def get_medians(df_dust, df_non_dust):
    median_dust = df_dust["moisture"].median(skipna=True)
    median_non_dust = df_non_dust["moisture"].median(skipna=True)

    medians = median_dust, median_non_dust

    return medians

In [ ]:
def plot_bar_chart_moisture(df_dust, df_non_dust):
    print("Plotting bar chart...")

    #--- Calculate bins
    bins = np.linspace(0, 0.4, 30)
    counts_dust, _ = np.histogram(df_dust["moisture"], bins=bins)
    counts_non_dust, _ = np.histogram(df_non_dust["moisture"], bins=bins)
    width = (bins[1] - bins[0]) / 3
    density_dust = counts_dust / np.sum(counts_dust)
    density_non_dust = counts_non_dust / np.sum(counts_non_dust)

    fig, ax_bar = plt.subplots(figsize=(12, 6))

    plt.bar(bins[:-1], density_dust, 
            width=width, 
            align='edge', 
            color="tab:orange",
            edgecolor="black",
            linewidth=1,
            label=f"dust events \n n={len(df_dust["moisture"]) :.2e}",)
    plt.bar(bins[:-1] + width, density_non_dust, 
            width=width, 
            align='edge', 
            color="tab:blue",
            label=f"non-dust grid \n n={len(df_non_dust["moisture"]) :.2e}",
            alpha=0.5)

    medians = get_medians(df_dust, df_non_dust)
    add_medians_to_plot(ax_bar, medians)
    ax_bar.set_ylabel("Fraction of total", fontsize=18)
    ax_bar.set_xlabel("Soil Moisture (0-10 cm) [m³/m³]", fontsize=18)
    ax_bar.set_title("Soil Moisture at Winds >= 10 m/s", fontsize=24)

    ax_bar.legend()
    plt.tight_layout()
    plt.show()

    return

In [ ]:
plot_bar_chart_moisture(dust_points_vars, non_dust_points_vars)